## Scraping

**Importing Libraries**

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urljoin

**Configuration Settings**

In [2]:
input_csv = "hardware_pioneers_swapcard_links_extracted.csv"          # your CSV file
output_csv = "scraped_results.csv"    # output file

profile_url = "profile_url"  # change this if your CSV column is called "URL", "links", "parsed_url", etc.

tags_to_search = ["h1", "h2", "h3", "p"]
class_name = "search_term"

REQUEST_DELAY = 0.5

tags_to_extract = ["h1", "h2", "h3", "h4", "p", "li", "a"]

In [3]:
def clean_text(text):
    if text:
        return re.sub(r'\s+', ' ', text).strip()
    return ""

URL Scraping Function

In [4]:
def scrape_url(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        
        response = requests.get(url, headers=headers, timeout=10)
       
        if response.status_code != 200:
            return {
                    "url": url,
                    "status": response.status_code,
                    "page_title": "",
                    "extracted_text": "",
                    "error": f"HTTP {response.status_code}"
            }
        
        soup = BeautifulSoup(response.content, "html.parser")

         # remove useless script/style text
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        page_title = soup.title.get_text(strip=True) if soup.title else ""

        elements = soup.find_all(["h1", "h2", "h3", "h4", "p", "li", "a"])

        extracted_parts = []

        for element in elements:
            text = element.get_text(" ", strip=True)
            text = clean_text(text)

            if text:
                extracted_parts.append(text)

        # remove duplicates
        extracted_parts = list(dict.fromkeys(extracted_parts))

        extracted_text = " | ".join(extracted_parts)

        return {
            "url": url,
            "status": response.status_code,
            "page_title": page_title,
            "extracted_text": extracted_text,
            "error": ""
        }

    except Exception as e:
        return {
            "url": url,
            "status": "",
            "page_title": "",
            "extracted_text": "",
            "error": str(e)
        }

Load URL from CSV

In [5]:
df = pd.read_csv(input_csv)

print("Columns found:")
print(df.columns.tolist())

profile_url = globals().get("profile_url", globals().get("url_columns", "profile_url"))
BASE_URL = globals().get("BASE_URL", "")

if profile_url not in df.columns:
    raise ValueError(f"Column '{profile_url}' not found. Available columns: {df.columns.tolist()}")

profile_urls = (
    df[profile_url]
    .dropna()
    .astype(str)
    .apply(lambda x: urljoin(BASE_URL, x))
    .drop_duplicates()
    .tolist()
)

print(f"Total URLs found: {len(profile_urls)}")

profile_urls[:5]

Columns found:
['company', 'stand_or_location', 'profile_url', 'raw_href', 'logo_proxy_url', 'logo_original_url']
Total URLs found: 50


['https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTQ=',
 'https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjYyNTE=',
 'https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTA=',
 'https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDY=',
 'https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDQ=']

Results

In [6]:
results = []

for i, profile_url in enumerate(profile_urls, start=1):
    print(f"Scraping {i}/{len(profile_urls)}: {profile_url}")

    result = scrape_url(profile_url)
    results.append(result)

    time.sleep(REQUEST_DELAY)

results_df = pd.DataFrame(results)

results_df.head()

Scraping 1/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTQ=
Scraping 2/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjYyNTE=
Scraping 3/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTA=
Scraping 4/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDY=
Scraping 5/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDQ=
Scraping 6/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDU=
Scraping 7/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTM=
Scraping 8/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTQ=
Scraping 9/50: https://app.swapcard.com/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJp

,url,status,page_title,extracted_text,error
0,https://app.swapcard.com/widget/event/hardware...,200,Esprit Electronics,Esprit Electronics | Information | Contact det...,
1,https://app.swapcard.com/widget/event/hardware...,200,PCBWay,PCBWay | Products & services | Team | Informat...,
2,https://app.swapcard.com/widget/event/hardware...,200,Win Source,"Win Source | Information | Since 1999, WIN SOU...",
3,https://app.swapcard.com/widget/event/hardware...,200,2J Antennas & Antenova,2J Antennas & Antenova | Information | 2J Ante...,
4,https://app.swapcard.com/widget/event/hardware...,200,3Point1 Developments,3Point1 Developments | Information | Working w...,


In [7]:
results_df.to_csv(output_csv, index=False)

print(f"Scraping completed. Results saved to {output_csv}")

Scraping completed. Results saved to scraped_results.csv


Extraction of Key Word Data from URLs

In [ ]:
#def extract_information(key_words):
#    keywords_data = []
#
#    for key in key_words:
#        keyword_data = []asdas
#        for keyword_data in key_words:
#            search_terms = key_words.find_all(
#                ["h1", "h2", "h3", "p"],
#                class_="search_term"
#            )
#
#
#        job_data = [
#            element.get_text(strip=True)
#            for element in search_terms
#        ]
#
#       keywords_data.append(keyword_data)
#
#    return keywords_data
#

SyntaxError: invalid syntax. Perhaps you forgot a comma? (307103803.py, line 5)

Extracting Text from inline HTML Classes

In [ ]:
#for job_card in job_cards:
#   print(job_card, end="\n" * 2)

Extracting Text from inline HTML Elements

In [ ]:
#search_term = [ "Graduate",
#               "Internship",
#               "Entry Level",
#               "Junior",
#               "Placement",]

#search_keywords = ["Artificial Intelligence", 
#                   "Data Scientist", 
#                   "Machine Learning", 
#                   "Engineer",]


In [ ]:
#def Extract_Information(job_cards):

#    fields = {
#    "search_term": "h1",
#    "search_term": "h2",
#    "search_term": "h3",
#    "search_term": "p"
#    }

#job_data = {}

#for key, tag in fields.items():
#    element = job_card.find(tag, class_="search_term")
#    job_data[key] = element.get_text(strip=True) if element else ""